# ReadBridge Nigeria: Book Catalogue Pipeline

Web scraping capstone, 10Alytics Data Engineering Programme

**Goal:** scrape 10 pages (200 books) from books.toscrape.com, clean the data with pandas, load it into a PostgreSQL table called `books_catalogue`, and query it back to confirm it worked.

**Pipeline:** Extract (requests + BeautifulSoup) -> Transform (pandas) -> Load (SQLAlchemy + PostgreSQL)

## 1. Setup and imports

In [1]:
# Run this once if the libraries are not installed yet
# %pip install requests beautifulsoup4 pandas sqlalchemy psycopg2-binary python-dotenv

In [2]:
import os
import time

import pandas as pd
import requests
from bs4 import BeautifulSoup
from dotenv import find_dotenv, load_dotenv
from sqlalchemy import create_engine, text
from sqlalchemy.engine import URL

print("Libraries imported")

Libraries imported


## 2. Scraper settings and a safe request function

The site has a `robots.txt` that allows scraping and it was built for practice, but I still wait one second between requests so the server is not overloaded.

In [3]:
BASE_URL = "https://books.toscrape.com/catalogue/page-{}.html"
TOTAL_PAGES = 10

RATING_MAP = {"One": 1, "Two": 2, "Three": 3, "Four": 4, "Five": 5}


def safe_get(url, delay=1):
    """Fetch a URL. Return the response, or None if the request fails."""
    try:
        response = requests.get(url, timeout=10)
        response.raise_for_status()
        # the site does not declare its encoding, so force UTF-8
        # (otherwise the pound sign shows up as "Â£")
        response.encoding = "utf-8"
        time.sleep(delay)
        return response
    except requests.RequestException as e:
        print(f"Request failed for {url}: {e}")
        return None

## 3. Look at one page first

Before looping, I check the structure of a single page. Every book sits inside an `<article class="product_pod">` tag.

In [4]:
resp = safe_get(BASE_URL.format(1))
soup = BeautifulSoup(resp.text, "html.parser")

books_on_page = soup.find_all("article", class_="product_pod")
print("Books on page 1:", len(books_on_page))
print(books_on_page[0].prettify()[:700])

Books on page 1: 20
<article class="product_pod">
 <div class="image_container">
  <a href="a-light-in-the-attic_1000/index.html">
   <img alt="A Light in the Attic" class="thumbnail" src="../media/cache/2c/da/2cdad67c44b002e7ead0cc35693c0e8b.jpg"/>
  </a>
 </div>
 <p class="star-rating Three">
  <i class="icon-star">
  </i>
  <i class="icon-star">
  </i>
  <i class="icon-star">
  </i>
  <i class="icon-star">
  </i>
  <i class="icon-star">
  </i>
 </p>
 <h3>
  <a href="a-light-in-the-attic_1000/index.html" title="A Light in the Attic">
   A Light in the ...
  </a>
 </h3>
 <div class="product_price">
  <p class="price_color">
   £51.77
  </p>
  <p class="instock availability">
   <i class="icon-ok">
   </i>
   I


## 4. Scrape pages 1 to 10

For each book I collect the title, price, rating (as a word like "Three"), availability, and the page number it came from.

In [5]:
def parse_book(book, page_number):
    """Pull the fields I need out of one <article> tag."""
    return {
        "title": book.find("h3").find("a")["title"],
        "price": book.find("p", class_="price_color").get_text(strip=True),
        # the rating is stored as a class name, e.g. class="star-rating Three"
        "rating": book.find("p", class_="star-rating")["class"][1],
        "availability": book.find("p", class_="availability").get_text(strip=True),
        "page_number": page_number,
    }


all_books = []

for page in range(1, TOTAL_PAGES + 1):
    url = BASE_URL.format(page)
    response = safe_get(url)

    if response is None:
        print(f"Skipping page {page}")
        continue

    soup = BeautifulSoup(response.text, "html.parser")
    cards = soup.find_all("article", class_="product_pod")

    for card in cards:
        all_books.append(parse_book(card, page))

    print(f"Page {page:>2}: {len(cards)} books scraped (running total: {len(all_books)})")

Page  1: 20 books scraped (running total: 20)
Page  2: 20 books scraped (running total: 40)
Page  3: 20 books scraped (running total: 60)
Page  4: 20 books scraped (running total: 80)
Page  5: 20 books scraped (running total: 100)
Page  6: 20 books scraped (running total: 120)
Page  7: 20 books scraped (running total: 140)
Page  8: 20 books scraped (running total: 160)
Page  9: 20 books scraped (running total: 180)
Page 10: 20 books scraped (running total: 200)


## 5. Raw DataFrame

Expecting 200 rows and 5 columns before any cleaning.

In [6]:
raw_df = pd.DataFrame(all_books)
print("Shape:", raw_df.shape)
raw_df.head(10)

Shape: (200, 5)


,title,price,rating,availability,page_number
0,A Light in the Attic,£51.77,Three,In stock,1
1,Tipping the Velvet,£53.74,One,In stock,1
2,Soumission,£50.10,One,In stock,1
3,Sharp Objects,£47.82,Four,In stock,1
4,Sapiens: A Brief History of Humankind,£54.23,Five,In stock,1
5,The Requiem Red,£22.65,One,In stock,1
6,The Dirty Little Secrets of Getting Your Dream...,£33.34,Four,In stock,1
7,The Coming Woman: A Novel Based on the Life of...,£17.93,Three,In stock,1
8,The Boys in the Boat: Nine Americans and Their...,£22.60,Four,In stock,1
9,The Black Maria,£52.15,One,In stock,1


In [7]:
raw_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 200 entries, 0 to 199
Data columns (total 5 columns):
 #   Column        Non-Null Count  Dtype 
---  ------        --------------  ----- 
 0   title         200 non-null    object
 1   price         200 non-null    object
 2   rating        200 non-null    object
 3   availability  200 non-null    object
 4   page_number   200 non-null    int64 
dtypes: int64(1), object(4)
memory usage: 7.9+ KB


## 6. Clean and transform

Cleaning steps:
1. `price`: remove the currency symbol and any encoding junk, then convert to a float
2. `rating`: map the words One to Five onto the numbers 1 to 5
3. `title` and `availability`: strip extra whitespace
4. Add a `scraped_at` timestamp, rounded down to the minute
5. Drop duplicates

In [8]:
df = raw_df.copy()

# 1. price: keep only digits and the decimal point, then convert to float
df["price"] = (
    df["price"]
    .str.replace(r"[^0-9.]", "", regex=True)
    .astype(float)
)

# 2. rating: word -> integer
df["rating"] = df["rating"].map(RATING_MAP).astype(int)

# 3. tidy up the text columns
df["title"] = df["title"].str.strip()
df["availability"] = df["availability"].str.strip()

# 4. timestamp of when the data was scraped (seconds and microseconds set to 0)
df["scraped_at"] = pd.Timestamp.now().replace(second=0, microsecond=0)

# 5. drop duplicate rows
before = len(df)
df = df.drop_duplicates().reset_index(drop=True)
print(f"Duplicates removed: {before - len(df)}")

print("Shape after cleaning:", df.shape)
df.head(10)

Duplicates removed: 0
Shape after cleaning: (200, 6)


,title,price,rating,availability,page_number,scraped_at
0,A Light in the Attic,51.77,3,In stock,1,2026-10-03 07:02:00
1,Tipping the Velvet,53.74,1,In stock,1,2026-10-03 07:02:00
2,Soumission,50.10,1,In stock,1,2026-10-03 07:02:00
3,Sharp Objects,47.82,4,In stock,1,2026-10-03 07:02:00
4,Sapiens: A Brief History of Humankind,54.23,5,In stock,1,2026-10-03 07:02:00
5,The Requiem Red,22.65,1,In stock,1,2026-10-03 07:02:00
6,The Dirty Little Secrets of Getting Your Dream...,33.34,4,In stock,1,2026-10-03 07:02:00
7,The Coming Woman: A Novel Based on the Life of...,17.93,3,In stock,1,2026-10-03 07:02:00
8,The Boys in the Boat: Nine Americans and Their...,22.60,4,In stock,1,2026-10-03 07:02:00
9,The Black Maria,52.15,1,In stock,1,2026-10-03 07:02:00


In [9]:
df.dtypes

title                   object
price                  float64
rating                   int64
availability            object
page_number              int64
scraped_at      datetime64[us]
dtype: object

## 7. Quality checks before loading

In [10]:
assert len(df) == 200, f"Expected 200 rows but found {len(df)}"
assert df["price"].notna().all(), "Some prices could not be converted"
assert df["rating"].between(1, 5).all(), "Rating outside the 1-5 range"
assert not df["title"].str.contains("Â", regex=False).any(), "Encoding artefacts left in titles"

print("Rows               :", len(df))
print("Missing values     :", int(df.isna().sum().sum()))
print("Price range        : £{:.2f} to £{:.2f}".format(df["price"].min(), df["price"].max()))
print("Rating distribution:")
print(df["rating"].value_counts().sort_index())

Rows               : 200
Missing values     : 0
Price range        : £10.16 to £59.64
Rating distribution:
rating
1    49
2    41
3    35
4    38
5    37
Name: count, dtype: int64


## 8. Load into PostgreSQL

Connection details: host `localhost`, port `5432`, database `readbridge_db`, user `postgres`.

The password is read from a `.env` file (variable `DB_PASSWORD`) that sits in the same folder as this notebook. The `.env` file is listed in `.gitignore`, so the password is never pushed to GitHub. The `readbridge_db` database must already exist (see the README).

In [11]:
# read DB_PASSWORD from the .env file in this folder
load_dotenv(find_dotenv(usecwd=True), override=True)
DB_PASSWORD = os.getenv("DB_PASSWORD")

if not DB_PASSWORD:
    raise ValueError("DB_PASSWORD not found. Create a .env file next to this notebook with the line: DB_PASSWORD=your_password")

db_url = URL.create(
    drivername="postgresql+psycopg2",
    username="postgres",
    password=DB_PASSWORD,
    host="localhost",
    port=5432,
    database="readbridge_db",
)

engine = create_engine(db_url)

# quick connection test
with engine.connect() as conn:
    print("Connected to:", conn.execute(text("SELECT current_database()")).scalar())

Connected to: readbridge_db


In [12]:
from sqlalchemy.types import Text, Numeric, Integer, DateTime

column_types = {
    "title": Text(),
    "price": Numeric(10, 2),
    "rating": Integer(),
    "availability": Text(),
    "page_number": Integer(),
    "scraped_at": DateTime(),
}

df.to_sql("books_catalogue", engine, if_exists="replace", index=False, dtype=column_types)
print("Data written to table: books_catalogue")

Data written to table: books_catalogue


## 9. Query it back to confirm

In [13]:
row_count = pd.read_sql("SELECT COUNT(*) AS total_rows FROM books_catalogue", engine)
row_count

,total_rows
0,200


In [14]:
pd.read_sql("SELECT * FROM books_catalogue LIMIT 10", engine)

,title,price,rating,availability,page_number,scraped_at
0,A Light in the Attic,51.77,3,In stock,1,2026-10-03 07:02:00
1,Tipping the Velvet,53.74,1,In stock,1,2026-10-03 07:02:00
2,Soumission,50.10,1,In stock,1,2026-10-03 07:02:00
3,Sharp Objects,47.82,4,In stock,1,2026-10-03 07:02:00
4,Sapiens: A Brief History of Humankind,54.23,5,In stock,1,2026-10-03 07:02:00
5,The Requiem Red,22.65,1,In stock,1,2026-10-03 07:02:00
6,The Dirty Little Secrets of Getting Your Dream...,33.34,4,In stock,1,2026-10-03 07:02:00
7,The Coming Woman: A Novel Based on the Life of...,17.93,3,In stock,1,2026-10-03 07:02:00
8,The Boys in the Boat: Nine Americans and Their...,22.60,4,In stock,1,2026-10-03 07:02:00
9,The Black Maria,52.15,1,In stock,1,2026-10-03 07:02:00


In [15]:
# check the column types PostgreSQL ended up with
pd.read_sql("""
    SELECT column_name, data_type
    FROM information_schema.columns
    WHERE table_name = 'books_catalogue'
    ORDER BY ordinal_position
""", engine)

,column_name,data_type
0,title,text
1,price,numeric
2,rating,integer
3,availability,text
4,page_number,integer
5,scraped_at,timestamp without time zone


In [16]:
# a few example business queries
pd.read_sql("""
    SELECT rating,
           COUNT(*)              AS num_books,
           ROUND(AVG(price), 2)  AS avg_price
    FROM books_catalogue
    GROUP BY rating
    ORDER BY rating DESC
""", engine)

,rating,num_books,avg_price
0,5,37,34.82
1,4,38,34.90
2,3,35,36.19
3,2,41,33.74
4,1,49,34.59


In [17]:
pd.read_sql("""
    SELECT title, price, rating
    FROM books_catalogue
    ORDER BY price DESC
    LIMIT 5
""", engine)

,title,price,rating
0,Thomas Jefferson and the Tripoli Pirates: The ...,59.64,1
1,The Death of Humanity: and the Case for Life,58.11,4
2,The White Cat and the Monk: A Retelling of the...,58.08,4
3,Immunity: How Elie Metchnikoff Changed the Cou...,57.36,5
4,I Had a Nice Time And Other Lies...: How to fi...,57.36,4


## 10. Summary

- Scraped 10 pages of books.toscrape.com (200 books) using a pagination loop with `time.sleep(1)` and a `safe_get()` function for error handling
- Cleaned `price` (text to numeric) and `rating` (word to integer), and added `page_number` and `scraped_at`
- Loaded the result into the `books_catalogue` table in PostgreSQL and confirmed the row count by querying it back